# PSOD: Advanced Tutorial

## Advanced Techniques for Outlier Detection

This notebook covers advanced topics including:

1. Custom base learners and ensemble methods
2. Advanced parameter tuning strategies
3. Feature engineering for better detection
4. Handling imbalanced datasets
5. Time series outlier detection
6. Interactive dashboards and advanced visualization
7. Performance optimization techniques

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# For development
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

from psod import (
    PSOD,
    evaluate_outlier_detection,
    compute_feature_importance,
    generate_outlier_data,
    calibrate_outlier_scores,
    combine_outlier_scores
)
from psod.visualization import (
    plot_outlier_scores,
    create_outlier_dashboard,
    create_interactive_explorer,
    plot_roc_pr_curves
)

print("Setup complete!")

## 1. Custom Base Learners

PSOD allows you to use different regression models as base learners. This can significantly improve performance depending on your data characteristics.

In [ ]:
# Generate complex dataset
np.random.seed(42)
n_samples = 500
n_outliers = 30

# Create non-linear relationships
X1 = np.random.randn(n_samples - n_outliers, 1)
X2 = X1 ** 2 + np.random.randn(n_samples - n_outliers, 1) * 0.1
X3 = np.sin(X1 * 2) + np.random.randn(n_samples - n_outliers, 1) * 0.1
X4 = np.exp(X1 * 0.5) + np.random.randn(n_samples - n_outliers, 1) * 0.5
X5 = np.random.randn(n_samples - n_outliers, 1)

normal_data = np.hstack([X1, X2, X3, X4, X5])

# Add outliers
outliers = np.random.uniform(-5, 5, (n_outliers, 5))
X = np.vstack([normal_data, outliers])
y_true = np.array([0] * (n_samples - n_outliers) + [1] * n_outliers)

df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(5)])

print(f"Dataset shape: {df.shape}")
print(f"Outliers: {n_outliers}")

In [ ]:
# Compare different base learners
from sklearn.linear_model import Ridge, Lasso

base_learners = {
    'Linear (default)': None,
    'Ridge': Ridge,
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor
}

results = {}

for name, learner in base_learners.items():
    print(f"\nTesting {name}...")
    
    if learner is None:
        detector = PSOD(min_cols_chosen=0.5, max_cols_chosen=1.0, random_seed=42)
    else:
        detector = PSOD(
            base_learner=learner,
            min_cols_chosen=0.5,
            max_cols_chosen=1.0,
            random_seed=42
        )
    
    scores = detector.fit_predict(df, return_class=False)
    labels = detector.fit_predict(df, return_class=True)
    
    metrics = evaluate_outlier_detection(y_true, labels, scores)
    results[name] = {'scores': scores, 'labels': labels, 'metrics': metrics}
    
    print(f"  F1-Score: {metrics['f1']:.3f}")
    print(f"  ROC-AUC: {metrics['roc_auc']:.3f}")

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    plot_outlier_scores(
        result['scores'],
        result['labels'],
        ax=axes[idx],
        title=f"{name}\nF1={result['metrics']['f1']:.3f}, AUC={result['metrics']['roc_auc']:.3f}"
    )

plt.tight_layout()
plt.show()

## 2. Advanced Parameter Tuning

Let's explore systematic parameter tuning using grid search.

In [ ]:
# Parameter grid
param_grid = {
    'stdevs_to_outlier': [1.5, 2.0, 2.5, 3.0],
    'min_cols_chosen': [0.3, 0.5, 0.7],
    'transform_algorithm': ['logarithmic', 'yeo-johnson', 'quantile', None]
}

best_score = 0
best_params = None
tuning_results = []

print("Starting parameter tuning...\n")

for stdev in param_grid['stdevs_to_outlier']:
    for min_cols in param_grid['min_cols_chosen']:
        for transform in param_grid['transform_algorithm']:
            detector = PSOD(
                stdevs_to_outlier=stdev,
                min_cols_chosen=min_cols,
                max_cols_chosen=1.0,
                transform_algorithm=transform,
                random_seed=42
            )
            
            scores = detector.fit_predict(df, return_class=False)
            labels = detector.fit_predict(df, return_class=True)
            
            metrics = evaluate_outlier_detection(y_true, labels, scores)
            f1 = metrics['f1']
            
            tuning_results.append({
                'stdevs_to_outlier': stdev,
                'min_cols_chosen': min_cols,
                'transform_algorithm': transform if transform else 'none',
                'f1_score': f1,
                'roc_auc': metrics['roc_auc']
            })
            
            if f1 > best_score:
                best_score = f1
                best_params = {
                    'stdevs_to_outlier': stdev,
                    'min_cols_chosen': min_cols,
                    'transform_algorithm': transform
                }

print(f"\nBest F1-Score: {best_score:.3f}")
print(f"Best Parameters: {best_params}\n")

# Display top 10 configurations
results_df = pd.DataFrame(tuning_results).sort_values('f1_score', ascending=False)
print("Top 10 Configurations:")
print(results_df.head(10))

In [ ]:
# Visualize parameter impact
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Impact of stdevs_to_outlier
grouped = results_df.groupby('stdevs_to_outlier')['f1_score'].mean()
axes[0].plot(grouped.index, grouped.values, 'o-', linewidth=2, markersize=10)
axes[0].set_xlabel('stdevs_to_outlier', fontsize=12)
axes[0].set_ylabel('Mean F1-Score', fontsize=12)
axes[0].set_title('Impact of stdevs_to_outlier', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Impact of min_cols_chosen
grouped = results_df.groupby('min_cols_chosen')['f1_score'].mean()
axes[1].plot(grouped.index, grouped.values, 's-', linewidth=2, markersize=10, color='orange')
axes[1].set_xlabel('min_cols_chosen', fontsize=12)
axes[1].set_ylabel('Mean F1-Score', fontsize=12)
axes[1].set_title('Impact of min_cols_chosen', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Impact of transformation
grouped = results_df.groupby('transform_algorithm')['f1_score'].mean().sort_values()
axes[2].barh(grouped.index, grouped.values, color='steelblue')
axes[2].set_xlabel('Mean F1-Score', fontsize=12)
axes[2].set_title('Impact of Transformation Algorithm', fontsize=14)
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 3. Ensemble Outlier Detection

Combine multiple PSOD detectors for improved robustness.

In [ ]:
# Train multiple detectors with different configurations
detectors = [
    PSOD(stdevs_to_outlier=2.0, min_cols_chosen=0.3, random_seed=42),
    PSOD(stdevs_to_outlier=2.5, min_cols_chosen=0.5, random_seed=43),
    PSOD(stdevs_to_outlier=3.0, min_cols_chosen=0.7, random_seed=44),
]

# Collect scores from each detector
ensemble_scores = {}

for idx, detector in enumerate(detectors):
    scores = detector.fit_predict(df, return_class=False)
    ensemble_scores[f'detector_{idx}'] = scores

print("Individual detector scores collected")

# Combine scores using different strategies
strategies = ['mean', 'max', 'median']
ensemble_results = {}

for strategy in strategies:
    combined_scores = combine_outlier_scores(ensemble_scores, method=strategy)
    
    # Threshold at mean + 2*std
    threshold = combined_scores.mean() + 2 * combined_scores.std()
    combined_labels = (combined_scores > threshold).astype(int)
    
    metrics = evaluate_outlier_detection(y_true, combined_labels, combined_scores)
    ensemble_results[strategy] = {
        'scores': combined_scores,
        'labels': combined_labels,
        'metrics': metrics
    }
    
    print(f"\n{strategy.capitalize()} combination:")
    print(f"  F1-Score: {metrics['f1']:.3f}")
    print(f"  ROC-AUC: {metrics['roc_auc']:.3f}")

# Compare with single best detector
best_single = PSOD(**best_params, random_seed=42)
best_scores = best_single.fit_predict(df, return_class=False)
best_labels = best_single.fit_predict(df, return_class=True)
best_metrics = evaluate_outlier_detection(y_true, best_labels, best_scores)

print(f"\nBest single detector:")
print(f"  F1-Score: {best_metrics['f1']:.3f}")
print(f"  ROC-AUC: {best_metrics['roc_auc']:.3f}")

## 4. Time Series Outlier Detection

Apply PSOD to time series data with temporal features.

In [ ]:
from datetime import datetime, timedelta

# Generate time series
np.random.seed(42)
n_points = 500
timestamps = pd.date_range(start='2024-01-01', periods=n_points, freq='H')

# Create trend and seasonality
trend = np.linspace(10, 30, n_points)
hours = np.array([t.hour for t in timestamps])
seasonality = 5 * np.sin(2 * np.pi * hours / 24)
noise = np.random.randn(n_points) * 0.5

values = trend + seasonality + noise

# Add anomalies
n_anomalies = 20
anomaly_indices = np.random.choice(n_points, n_anomalies, replace=False)
y_true_ts = np.zeros(n_points)

for idx in anomaly_indices:
    values[idx] += np.random.choice([-1, 1]) * np.random.uniform(10, 20)
    y_true_ts[idx] = 1

# Create DataFrame
df_ts = pd.DataFrame({
    'timestamp': timestamps,
    'value': values
})

print(f"Time series shape: {df_ts.shape}")
print(f"Anomalies: {n_anomalies}")

In [ ]:
# Feature engineering for time series
def create_temporal_features(df, value_col='value'):
    df = df.copy()
    
    # Time features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    
    # Cyclic encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Lag features
    for lag in [1, 2, 3, 6, 12, 24]:
        df[f'lag_{lag}'] = df[value_col].shift(lag)
    
    # Rolling statistics
    for window in [3, 6, 12, 24]:
        df[f'rolling_mean_{window}'] = df[value_col].rolling(window=window).mean()
        df[f'rolling_std_{window}'] = df[value_col].rolling(window=window).std()
    
    # Difference
    df['diff_1'] = df[value_col].diff(1)
    df['diff_24'] = df[value_col].diff(24)
    
    return df.fillna(method='bfill').fillna(method='ffill')

df_ts_features = create_temporal_features(df_ts)

# Select features for detection
feature_cols = [col for col in df_ts_features.columns if col not in ['timestamp']]
X_ts = df_ts_features[feature_cols]

print(f"Features created: {len(feature_cols)}")

In [ ]:
# Detect anomalies
detector_ts = PSOD(
    min_cols_chosen=0.5,
    max_cols_chosen=1.0,
    stdevs_to_outlier=2.5,
    transform_algorithm='yeo-johnson',
    random_seed=42
)

scores_ts = detector_ts.fit_predict(X_ts, return_class=False)
labels_ts = detector_ts.fit_predict(X_ts, return_class=True)

metrics_ts = evaluate_outlier_detection(y_true_ts, labels_ts, scores_ts)

print(f"\nTime series detection results:")
print(f"  Detected: {sum(labels_ts)} anomalies")
print(f"  F1-Score: {metrics_ts['f1']:.3f}")
print(f"  ROC-AUC: {metrics_ts['roc_auc']:.3f}")

In [ ]:
# Visualize time series with anomalies
from psod.visualization import plot_timeseries_outliers

fig, ax = plt.subplots(figsize=(16, 6))
plot_timeseries_outliers(
    df_ts['timestamp'].values,
    df_ts['value'].values,
    labels_ts,
    scores_ts,
    ax=ax
)
plt.tight_layout()
plt.show()

## 5. Interactive Dashboard

Create an interactive dashboard for exploring outlier detection results.

In [ ]:
# Create comprehensive dashboard
fig = create_outlier_dashboard(
    df,
    scores,
    labels,
    feature_names=df.columns.tolist(),
    y_true=y_true
)
plt.show()

print("Dashboard created!")

## 6. Advanced Visualization: ROC and PR Curves

Compare multiple detectors using ROC and Precision-Recall curves.

In [ ]:
# Collect scores from different methods
all_scores = {
    'PSOD (default)': results['Linear (default)']['scores'],
    'PSOD (RF)': results['Random Forest']['scores'],
    'PSOD (GB)': results['Gradient Boosting']['scores'],
    'Ensemble (mean)': ensemble_results['mean']['scores']
}

# Plot ROC and PR curves
fig = plot_roc_pr_curves(y_true, all_scores, figsize=(14, 6))
plt.show()

## 7. Performance Optimization Tips

### Tips for large datasets:

1. **Use `sample_frac`**: Sample a subset of data for training
2. **Reduce `max_cols_chosen`**: Use fewer features per model
3. **Set `n_jobs=-1`**: Use all CPU cores
4. **Choose simpler base learners**: Linear models are faster than tree-based
5. **Use appropriate transformations**: Some are faster than others

In [ ]:
import time

# Generate large dataset
np.random.seed(42)
n_large = 5000
X_large = np.random.randn(n_large, 20)
df_large = pd.DataFrame(X_large)

print(f"Large dataset: {df_large.shape}\n")

# Compare performance
configs = [
    {'name': 'Default', 'params': {}},
    {'name': 'Sampling', 'params': {'sample_frac': 0.5}},
    {'name': 'Fewer features', 'params': {'max_cols_chosen': 0.5}},
    {'name': 'Optimized', 'params': {'sample_frac': 0.7, 'max_cols_chosen': 0.5}}
]

for config in configs:
    detector = PSOD(
        min_cols_chosen=0.3,
        stdevs_to_outlier=2.0,
        random_seed=42,
        **config['params']
    )
    
    start = time.time()
    scores = detector.fit_predict(df_large, return_class=False)
    elapsed = time.time() - start
    
    print(f"{config['name']:15s}: {elapsed:.2f}s")

## Summary

In this advanced tutorial, we covered:

1. ✅ Using custom base learners for improved performance
2. ✅ Systematic parameter tuning with grid search
3. ✅ Ensemble methods for robust detection
4. ✅ Time series outlier detection with temporal features
5. ✅ Interactive dashboards for exploration
6. ✅ Advanced evaluation with ROC/PR curves
7. ✅ Performance optimization for large datasets

### Key Takeaways:

- **Tree-based models** work well for non-linear relationships
- **Parameter tuning** can significantly improve results
- **Ensemble methods** provide robustness
- **Feature engineering** is crucial for time series
- **Sampling strategies** help with large datasets

### Next Steps:

Check out the **Real-World Case Studies** notebook for practical applications!